# 1. DATA PIPELINE & FEATURE ENGINEERING (HỌC MÁY)
*Phụ trách: Nguyễn Công Tiến*

Notebook này phụ trách việc cào dữ liệu chứng khoán (ví dụ mã FPT), làm sạch và lưu thành file CSV để làm dữ liệu chuẩn cho các mô hình Học Sâu và Học Máy.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import os

# Cấu hình thư mục lưu trữ
DATA_DIR = '../01_Data'
os.makedirs(DATA_DIR, exist_ok=True)

# Thông số tải dữ liệu
TICKER = 'FPT.VN' # Mã cổ phiếu FPT trên sàn chứng khoán Việt Nam (thông qua yfinance)
START_DATE = '2021-01-01'
END_DATE = '2026-05-01'

print(f"Đang tải dữ liệu {TICKER} từ {START_DATE} đến {END_DATE}...")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Tải dữ liệu
df = yf.download(TICKER, start=START_DATE, end=END_DATE)

if df.empty:
    print("Lỗi: Không lấy được dữ liệu. Có thể do kết nối mạng hoặc mã chứng khoán không đúng.")
    # Fallback tạo dữ liệu giả lập nếu không tải được (để đảm bảo code luôn chạy qua được)
    dates = pd.date_range(start=START_DATE, end=END_DATE, freq='B')
    df = pd.DataFrame(np.random.randn(len(dates), 5), index=dates, columns=['Open', 'High', 'Low', 'Close', 'Volume'])
    df['Close'] = df['Close'].cumsum() + 100 # Tạo đường giá có xu hướng đi lên
else:
    print(f"Tải thành công {len(df)} dòng dữ liệu.")

# Hiển thị 5 dòng đầu
df.head()

In [ ]:
# Tiền xử lý đơn giản: Xử lý missing value
df = df.dropna()

# Trực quan hóa giá đóng cửa
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label=f'Giá đóng cửa {TICKER}')
plt.title(f'Biểu đồ giá cổ phiếu {TICKER} (2021-2026)')
plt.xlabel('Thời gian')
plt.ylabel('Giá')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Tính toán biến động lợi suất (Dùng cho Machine Learning)
df['Return'] = df['Close'].pct_change()
df['Target'] = np.where(df['Return'] > 0, 1, 0) # 1 là Tăng, 0 là Giảm
df = df.dropna()

# Lưu dữ liệu sạch ra CSV để LSTM và Random Forest xài chung
csv_path = os.path.join(DATA_DIR, f'{TICKER.replace(".", "_")}_clean.csv')
df.to_csv(csv_path)
print(f"Đã lưu dữ liệu sạch tại: {csv_path}")